# GNN grid on Colab (Phase 3, T6)

Runs the `src/gnn.py` grid on a Colab GPU, unattended. **Before the first run:** in Google Drive create `MyDrive/aml/` and upload `data/gnn/edges.npz` and `data/gnn/head.npy` from the laptop into it (about 1.3 GB). Runtime → Change runtime type → **T4 GPU**.

Everything a run produces goes straight to `MyDrive/aml/runs/` (scores, weights, epoch history) and its live log to `MyDrive/aml/logs/<tag>.log`. Runs that already have a result in Drive are skipped, so after a disconnect just run all cells again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/aml'
!mkdir -p {DRIVE}/runs {DRIVE}/logs && ls -la {DRIVE}

Clone the `gnn` branch. The repo is private: add a GitHub token (fine-grained, read-only access to this repo) in Colab's **Secrets** panel (key icon on the left) under the name `GITHUB_TOKEN`, and allow this notebook to use it.

In [ ]:
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
!rm -rf /content/AML && git clone -q -b gnn https://{TOKEN}@github.com/trietsim69-boop/AML.git /content/AML
%cd /content/AML
!git log --oneline -1
# Colab already has torch, numpy, pandas, scikit-learn, duckdb and psutil; only PyG is missing
!pip install -q torch_geometric
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch the runtime to T4')

In [ ]:
# data from Drive; runs/ is a link into Drive so results survive a disconnect
!mkdir -p data/gnn && cp {DRIVE}/edges.npz {DRIVE}/head.npy data/gnn/ && ln -sfn {DRIVE}/runs data/gnn/runs && ls -la data/gnn

Smoke test: 3 hours per split, 1 epoch. Should finish in about a minute and print a `best val PR-AUC` line.

In [ ]:
!GNN_DEVICE=cuda python -u src/gnn.py train smoke max_hours=3 max_epochs=1 2>&1 | grep -v -i warn
!rm -f {DRIVE}/runs/smoke.*

The grid. Defaults are IBM Multi-GNN's GIN settings (`DEFAULTS` in `src/gnn.py`): hidden 64, lr 0.006, positive weight 6, dropout 0.01 / 0.1, batches of 8,192, up to 100 epochs with patience 10. Each row overrides some of them. Progress: open `MyDrive/aml/logs/<tag>.log`, or run the last cell.

In [ ]:
import json, os, subprocess, time
GRID = [
    ('gnn_ibm',      ''),                       # IBM defaults, 24h window
    ('gnn_pw10',     'pos_weight=10'),
    ('gnn_w72',      'window_h=72'),            # 3-day window, the longest our rules allow
    ('mlp_ibm',      'name=mlp'),               # the control under the same settings
]
for tag, extra in GRID:
    if os.path.exists(f'{DRIVE}/runs/{tag}.json'):
        print(f'{tag}: already done, skipped'); continue
    t0 = time.time(); print(f'{tag}: started {time.strftime("%H:%M")}, log {DRIVE}/logs/{tag}.log', flush=True)
    with open(f'{DRIVE}/logs/{tag}.log', 'w') as log:
        rc = subprocess.run(f'python -u src/gnn.py train {tag} {extra}', shell=True, stdout=log, stderr=subprocess.STDOUT,
                            env={**os.environ, 'GNN_DEVICE': 'cuda'}).returncode
    if rc: print(f'{tag}: FAILED (exit {rc}), see the log'); continue
    r = json.load(open(f'{DRIVE}/runs/{tag}.json'))
    print(f"{tag}: best val PR-AUC {r['best_val_pr_auc']:.4f} at epoch {r['best_epoch']} of {r['epochs_run']}, {(time.time()-t0)/60:.0f} min", flush=True)

Seeds of the best config: fill in `BEST` (its overrides from the grid above) once the grid has finished.

In [ ]:
BEST = ''          # e.g. 'window_h=72'
for seed in (1, 2):
    for name in ('gnn', 'mlp'):
        tag = f'{name}_best_s{seed}'
        if os.path.exists(f'{DRIVE}/runs/{tag}.json'): continue
        with open(f'{DRIVE}/logs/{tag}.log', 'w') as log:
            subprocess.run(f'python -u src/gnn.py train {tag} name={name} seed={seed} {BEST}', shell=True, stdout=log,
                           stderr=subprocess.STDOUT, env={**os.environ, 'GNN_DEVICE': 'cuda'})
        print(tag, 'done', flush=True)

Progress and summary at any time.

In [ ]:
!for f in {DRIVE}/logs/*.log; do echo "== $f"; grep -v -i warn "$f" | tail -3; done
!python src/gnn.py runs && cp reports/phase3-gnn-runs.csv {DRIVE}/